# 10 — Formula Deep Dive Compendium (Deep Learning Module)

This notebook is the math-first companion to lessons `00` through `09` in `deep learning/`.

Each formula section follows the same structure:
- Formal definition
- Symbols and parameters (word-by-word style)
- Assumptions
- Why used / effect on model quality
- Research context (canonical paper/source + practical takeaway)
- Mathematical advantages
- Mathematical disadvantages / failure modes


## Coverage

| Notebook | Core formulas covered in this compendium |
|---|---|
| 00_baseline | ERM objective, mini-batch backprop + SGD update |
| 01_weight_initialization | variance propagation, Xavier/He scaling laws |
| 02_batch_normalization | batch normalization train/inference transforms |
| 03_dropout | inverted dropout estimator, stochastic model averaging view |
| 04_regularization | L1/L2 penalized objective, shrinkage/proximal updates |
| 05_early_stopping | patience stopping rule and checkpoint selection |
| 06_optuna_hyperparameter_tuning | validation-risk minimization, TPE acquisition ratio |
| 07_xgboost_lightgbm | additive boosting update, second-order split gain |
| 08_feature_engineering | feature map composition, bias-variance, target encoding smoothing |
| 09_kaggle_competition_playbook | private-metric objective, CV proxy, weighted ensembling |


## Formula 00.1 — Empirical risk minimization (ERM)

### Formal definition
$$\mathcal{R}(\theta)=\mathbb{E}_{(x,y)\sim\mathcal{D}}[\ell(f_\theta(x),y)],\quad \hat{\mathcal{R}}_n(\theta)=\frac{1}{n}\sum_{i=1}^n \ell(f_\theta(x_i),y_i),\quad \theta^*=\arg\min_{\theta}\hat{\mathcal{R}}_n(\theta).$$

### Symbols and parameters (word-by-word style)
- $\mathcal{D}$: unknown data-generating distribution.
- $f_\theta$: model parameterized by $\theta$.
- $\ell(\cdot,\cdot)$: per-example loss function.
- $\mathcal{R}(\theta)$: population risk (true expected error).
- $\hat{\mathcal{R}}_n(\theta)$: empirical risk on $n$ training points.
- $\theta^*$: ERM optimizer in hypothesis space.

### Assumptions
- Training and test examples are sampled i.i.d. from the same distribution.
- The empirical average is a useful estimator of expectation.
- The function class is expressive enough to capture signal.

### Why used / effect on model quality
- Turns learning into a tractable optimization problem from finite data.
- Directly controls training objective used in all downstream techniques.
- Better ERM optimization generally improves calibration and ranking quality before regularization trade-offs.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Vapnik's statistical learning framework (1992, 1998).
- Practical takeaway: minimizing training loss is necessary but not sufficient; generalization controls (regularization, validation) are required.

### Mathematical advantages
- Convex for many classical models, enabling global guarantees.
- Uniform interface across regression, classification, and ranking losses.
- Supports concentration bounds linking empirical and population error.

### Mathematical disadvantages / failure modes
- Over-parameterized models can drive $\hat{\mathcal{R}}_n \to 0$ while test error worsens.
- Distribution shift breaks the i.i.d.-based approximation.
- Loss mismatch with leaderboard metric can optimize the wrong target.


## Formula 00.2 — Mini-batch backpropagation and SGD step

### Formal definition
$$
g_t = \frac{1}{m} \sum_{i \in \mathcal{B}_t}
\nabla_\theta \ell(f_\theta(x_i), y_i), \quad
\nabla_\theta \ell =
\frac{\partial \ell}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial h_L} \cdots
\frac{\partial h_1}{\partial \theta}, \quad
\theta_{t+1} = \theta_t - \eta \left( g_t + \nabla_\theta \Omega(\theta_t) \right)
$$

### Symbols and parameters (word-by-word style)
- $\mathcal{B}_t$: mini-batch sampled at iteration $t$.
- $m$: batch size.
- $g_t$: stochastic gradient estimate.
- $h_1,\ldots,h_L$: hidden representations through depth $L$.
- $\eta$: learning rate.
- $\Omega(\theta)$: optional regularization penalty.

### Assumptions
- Mini-batch gradient is approximately unbiased for full gradient.
- Chain-rule factors are numerically stable enough for backpropagation.
- Learning rate is chosen inside a stable region.

### Why used / effect on model quality
- Enables scalable optimization on large datasets and deep nets.
- Gradient noise from mini-batching can help escape sharp basins.
- Correct update scaling strongly affects convergence speed and final validation score.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Rumelhart, Hinton, Williams (1986) for backprop; Robbins and Monro (1951) for stochastic approximation.
- Practical takeaway: stable gradient flow and learning-rate schedules dominate baseline training outcomes.

### Mathematical advantages
- Per-step cost is $O(m)$ instead of $O(n)$.
- Works with arbitrary differentiable architectures via automatic differentiation.
- Noise can act as implicit regularization.

### Mathematical disadvantages / failure modes
- High gradient variance slows or destabilizes convergence.
- Exploding/vanishing chain-rule products cause optimization pathologies.
- Poorly tuned $\eta$ leads to divergence or excessive underfitting.


## Formula 01.1 — Variance propagation through layers

### Formal definition
$$\mathrm{Var}[a_l] \approx n_{in}\,\mathrm{Var}[W_l] \,\mathrm{Var}[h_{l-1}],\quad a_l=W_l h_{l-1}. $$

### Symbols and parameters (word-by-word style)
- $a_l$: pre-activation vector at layer $l$.
- $W_l$: weight matrix for layer $l$.
- $h_{l-1}$: previous-layer activations.
- $n_{in}$: fan-in (input width to the layer).
- $\mathrm{Var}[\cdot]$: variance operator.

### Assumptions
- Weight entries are independent, zero-mean, and identically distributed.
- Inputs and weights are approximately independent at initialization.
- Finite second moments exist.

### Why used / effect on model quality
- Quantifies when activations blow up or collapse across depth.
- Guides initialization scaling so gradients remain informative.
- Better-conditioned depth improves both convergence speed and final accuracy.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Glorot and Bengio (2010) signal propagation analysis.
- Practical takeaway: preserving activation variance layer-by-layer is foundational for training deep MLPs.

### Mathematical advantages
- Gives closed-form scaling rule tied to architecture width.
- Connects forward signal stability to backward gradient stability.
- Simple diagnostic for initialization quality.

### Mathematical disadvantages / failure modes
- Independence assumptions break after a few updates.
- Nonlinearities with heavy truncation (e.g., saturated activations) violate simple variance approximations.
- Ignores covariance structure and higher moments.


## Formula 01.2 — Xavier and He initialization laws

### Formal definition
$$
W_{ij}^{\text{Xavier}} \sim \mathcal{U} \left(
-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}},
\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}
\right), \quad
W_{ij}^{\text{He}} \sim \mathcal{N} \left(
0, \frac{2}{n_{\text{in}}}
\right)
$$

### Symbols and parameters (word-by-word style)
- $W_{ij}$: weight connecting input unit $j$ to output unit $i$.
- $n_{in}, n_{out}$: fan-in and fan-out.
- $\mathcal{U}(a,b)$: uniform distribution on $[a,b]$.
- $\mathcal{N}(0,\sigma^2)$: zero-mean Gaussian with variance $\sigma^2$.

### Assumptions
- Xavier is intended for symmetric activations (e.g., tanh-like regimes).
- He scaling is tailored for ReLU-family activations where half-mass is truncated.
- Biases are initialized near zero.

### Why used / effect on model quality
- Places network in a trainable regime at step zero.
- Reduces epochs wasted on correcting poor initial scale.
- Improves reproducibility by lowering initialization-induced fold variance.

### Research context (canonical paper/source + practical takeaway)
- Canonical sources: Glorot and Bengio (2010), He et al. (2015).
- Practical takeaway: activation-aware initialization is one of the highest-leverage low-cost changes in deep learning pipelines.

### Mathematical advantages
- Explicit variance-preserving formulas.
- Compatible with standard dense and convolutional layers.
- Improves gradient norm homogeneity across depth.

### Mathematical disadvantages / failure modes
- Can still fail in very deep residual-free stacks.
- Not sufficient when data scaling is poor.
- Assumes stationarity that quickly degrades during training.


## Formula 02.1 — Batch normalization (training transform)

### Formal definition
$$\mu_\mathcal{B}=\frac{1}{m}\sum_{i=1}^m a_i,\quad \sigma_\mathcal{B}^2=\frac{1}{m}\sum_{i=1}^m(a_i-\mu_\mathcal{B})^2,\quad \hat{a}_i=\frac{a_i-\mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2+\epsilon}},\quad y_i=\gamma\hat{a}_i+\beta.$$

### Symbols and parameters (word-by-word style)
- $a_i$: pre-activation for sample $i$ in the batch.
- $\mu_\mathcal{B},\sigma_\mathcal{B}^2$: batch mean and variance.
- $\epsilon$: numerical stabilizer.
- $\hat{a}_i$: normalized activation.
- $\gamma,\beta$: learned scale and shift.

### Assumptions
- Batch statistics approximate local population moments.
- Batch size is large enough to keep noise manageable.
- Features are normalized independently per channel/unit.

### Why used / effect on model quality
- Smooths optimization landscape by re-centering and re-scaling activations.
- Permits larger learning rates and faster convergence.
- Often improves validation quality through optimization stabilization.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Ioffe and Szegedy (2015).
- Practical takeaway: BN is a practical optimizer aid first, regularizer second; tiny batches often weaken its benefits.

### Mathematical advantages
- Reduces internal covariate shift effects in practice.
- Learnable affine terms preserve representational capacity.
- Improves gradient conditioning in deep stacks.

### Mathematical disadvantages / failure modes
- Batch-statistic noise hurts performance for very small or non-i.i.d. batches.
- Train/inference discrepancy can create calibration drift.
- Sensitive to mixed-distribution mini-batches.


## Formula 02.2 — Batch normalization (inference with running statistics)

### Formal definition
$$\mu_{run}\leftarrow (1-\alpha)\mu_{run}+\alpha\mu_\mathcal{B},\quad \sigma_{run}^2\leftarrow (1-\alpha)\sigma_{run}^2+\alpha\sigma_\mathcal{B}^2,\quad y=\gamma\frac{a-\mu_{run}}{\sqrt{\sigma_{run}^2+\epsilon}}+\beta.$$

### Symbols and parameters (word-by-word style)
- $\mu_{run},\sigma_{run}^2$: exponential moving averages used at inference.
- $\alpha$: momentum/update rate for running moments.
- $a$: inference-time pre-activation.
- $y$: normalized and affine-transformed output.

### Assumptions
- Running moments converge to representative population statistics.
- Deployment distribution is close to training distribution.
- Momentum hyperparameter balances responsiveness and variance.

### Why used / effect on model quality
- Makes predictions deterministic at inference.
- Removes dependence on concurrent batch composition at test time.
- Good running-stat estimates reduce train-test normalization gap.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Ioffe and Szegedy (2015), plus deployment practice in modern deep-learning frameworks.
- Practical takeaway: mismatched running stats are a frequent hidden cause of degraded leaderboard performance.

### Mathematical advantages
- Exponential averaging gives low-memory online moment estimation.
- Deterministic transform simplifies reproducibility.
- Compatible with mixed hardware/inference pipelines.

### Mathematical disadvantages / failure modes
- Distribution drift invalidates stored moments.
- Poorly warmed statistics early in training can bias validation.
- Momentum choice introduces bias-variance trade-off in moment estimates.


## Formula 03.1 — Inverted dropout estimator

### Formal definition
$$m_j\sim \mathrm{Bernoulli}(1-p),\quad \tilde{h}_j=\frac{m_j}{1-p}h_j,\quad \mathbb{E}[\tilde{h}_j]=h_j.$$

### Symbols and parameters (word-by-word style)
- $h_j$: pre-dropout hidden activation for unit $j$.
- $m_j$: random binary keep-mask.
- $p$: dropout probability.
- $\tilde{h}_j$: post-dropout scaled activation.

### Assumptions
- Mask draws are independent across units (and often samples).
- Keep probability $1-p$ is not too small for usable signal.
- Train-time stochasticity is removed at inference.

### Why used / effect on model quality
- Prevents co-adaptation by randomly removing features during training.
- Acts as stochastic regularization that often narrows train-validation gap.
- Inverted scaling keeps expected activation magnitude stable.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Srivastava et al. (2014).
- Practical takeaway: moderate dropout can improve generalization, but aggressive dropout can underfit tabular MLPs.

### Mathematical advantages
- Unbiased activation estimator under mask randomness.
- Simple closed-form scaling.
- Provides implicit model averaging over sub-networks.

### Mathematical disadvantages / failure modes
- Increases gradient variance and may slow convergence.
- Interacts poorly with tiny networks or low-signal features.
- Can degrade calibration if dropout rate is mis-specified.


## Formula 03.2 — Dropout as stochastic model averaging

### Formal definition
$$
\min_\theta \; \mathbb{E}_{m \sim q_p(m)} \left[
\ell \left( f_{\theta,m}(x), y \right) \right], \quad
f_{\text{test}}(x) \approx \mathbb{E}_{m \sim q_p(m)} \left[ f_{\theta,m}(x) \right]
$$

### Symbols and parameters (word-by-word style)
- $m$: full dropout mask across all dropout layers.
- $q_p(m)$: mask distribution induced by dropout rate $p$.
- $f_{\theta,m}$: sub-network defined by mask $m$.
- $f_{test}$: deterministic test-time predictor.

### Assumptions
- Deterministic test network approximates ensemble mean.
- Loss and network nonlinearity allow practical Monte-Carlo approximation.
- Mask-induced subnetworks share parameters effectively.

### Why used / effect on model quality
- Reframes dropout as approximate ensembling without storing many models.
- Typically lowers variance and improves robustness on noisy features.
- Helps avoid brittle reliance on any single pathway.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Srivastava et al. (2014); Bayesian interpretation in Gal and Ghahramani (2016).
- Practical takeaway: dropout can be viewed as cheap ensemble regularization and uncertainty approximation.

### Mathematical advantages
- Ensemble-like variance reduction at near-single-model memory cost.
- Connects naturally to Bayesian approximate inference narratives.
- Encourages distributed representations.

### Mathematical disadvantages / failure modes
- Approximation error between deterministic scaling and true ensemble can be nontrivial.
- Adds stochastic noise that complicates hyperparameter tuning.
- Benefits can diminish with strong data augmentation or heavy label noise.


## Formula 04.1 — L1/L2-regularized objective

### Formal definition
$$
\mathcal{J}(\theta) = \mathcal{L}(\theta) + \lambda_2 \lVert \theta \rVert_2^2 + \lambda_1 \lVert \theta \rVert_1, \quad
\nabla_\theta \mathcal{J} = \nabla_\theta \mathcal{L} + 2 \lambda_2 \theta + \lambda_1 \, \mathrm{sign}(\theta)
$$

### Symbols and parameters (word-by-word style)
- $\mathcal{L}(\theta)$: data-fit loss term.
- $\lambda_2$: L2 penalty coefficient.
- $\lambda_1$: L1 penalty coefficient.
- $\lVert \theta \rVert_2^2$: squared Euclidean norm.
- $\lVert\theta\rVert_1$: absolute-value norm.

### Assumptions
- Penalized objective remains aligned with downstream metric.
- Regularization strengths are tuned on validation, not test.
- Parameter scaling is comparable across coordinates.

### Why used / effect on model quality
- Controls overfitting by discouraging overly complex parameterizations.
- L2 favors smooth shrinkage; L1 encourages sparsity and feature selection.
- Often reduces validation variance in medium-size tabular settings.

### Research context (canonical paper/source + practical takeaway)
- Canonical sources: Tikhonov regularization (1943), Tibshirani's Lasso (1996), elastic-net lineage (Zou and Hastie, 2005).
- Practical takeaway: mixing L1/L2 gives a robust bias-variance knob when model capacity is high relative to data size.

### Mathematical advantages
- Improves conditioning and can reduce estimator variance.
- L1 yields sparse solutions interpretable at feature level.
- Convex in linear settings with strong theoretical guarantees.

### Mathematical disadvantages / failure modes
- Excessive penalty causes underfitting and systematic bias.
- L1 subgradients are unstable near zero in noisy settings.
- Improper feature scaling skews penalty fairness.


## Formula 04.2 — Shrinkage dynamics (weight decay and proximal L1)

### Formal definition
$$
\theta_{t+1}^{(L2)} = (1 - 2 \eta \lambda_2) \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t), \quad
u_t = \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t), \quad
\theta_{t+1,j}^{(L1)} = \mathrm{sign}(u_{t,j}) \max\left( |u_{t,j}| - \eta \lambda_1, 0 \right)
$$

### Symbols and parameters (word-by-word style)
- $\eta$: learning rate.
- $u_t$: gradient step before proximal shrinkage.
- $\theta_{t+1}^{(L2)}$: parameter after decoupled L2 shrinkage.
- $\theta_{t+1,j}^{(L1)}$: coordinate-wise soft-thresholded update.

### Assumptions
- Step size is small enough for stable explicit updates.
- L2 expression matches SGD-style weight decay approximation.
- Proximal interpretation is applied coordinate-wise.

### Why used / effect on model quality
- Explains how regularization acts during optimization, not only in objective space.
- L2 continuously contracts weights; L1 can zero out weak coordinates.
- Understanding update geometry helps tune $\lambda_1,\lambda_2$ more reliably.

### Research context (canonical paper/source + practical takeaway)
- Canonical sources: proximal gradient methods (Parikh and Boyd, 2014) and decoupled weight decay insights (Loshchilov and Hutter, 2019).
- Practical takeaway: optimizer-specific regularization behavior matters; identical coefficients can behave differently across optimizers.

### Mathematical advantages
- Provides explicit trajectory-level intuition.
- Soft-threshold map gives exact sparsity condition.
- Clarifies regularization-strength and learning-rate coupling.

### Mathematical disadvantages / failure modes
- Coupled decay in adaptive optimizers can deviate from this simple form.
- Large $\eta\lambda$ can cause overly aggressive contraction.
- Non-smooth L1 dynamics can hinder convergence diagnostics.


## Formula 05.1 — Early stopping rule with patience

### Formal definition
$$\min_{k\in[t-P+1,t]} v_k>v_{best}\;\Rightarrow\;\text{stop at epoch }t,\quad t^*=\arg\min_t v_t,\quad \theta_{deploy}=\theta_{t^*}. $$

### Symbols and parameters (word-by-word style)
- $v_t$: validation metric (loss or error) at epoch $t$.
- $P$: patience window length.
- $v_{best}$: best validation value observed so far.
- $t^*$: best epoch index.
- $\theta_{deploy}$: checkpoint restored for inference.

### Assumptions
- Validation split is representative of unseen test data.
- Metric noise is not so high that patience logic is purely random.
- Checkpointing correctly restores optimizer/model state as needed.

### Why used / effect on model quality
- Stops training near the minimum validation risk before overfitting dominates.
- Converts epoch count into a data-dependent regularization mechanism.
- Often improves private-score stability relative to fixed-epoch training.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: early-stopping theory for iterative regularization (Prechelt, 1998; Yao et al., 2007).
- Practical takeaway: patience and minimum-delta settings are high-impact hyperparameters for noisy Kaggle loops.

### Mathematical advantages
- Zero-cost regularization using existing validation measurements.
- Adapts training horizon to optimization dynamics.
- Easy to combine with checkpoint averaging or ensembling.

### Mathematical disadvantages / failure modes
- Noisy validation curves can trigger premature stopping.
- Biased validation split leads to wrong stopping epoch.
- Repeated peeking may overfit to validation if tuning is excessive.


## Formula 06.1 — Hyperparameter selection as outer optimization

### Formal definition
$$
\lambda^* = \arg\min_{\lambda \in \Lambda} \mathcal{V}(\lambda), \quad
\mathcal{V}(\lambda) = \frac{1}{K} \sum_{k=1}^K \mathcal{M}^{(k)} \left( \theta^*(\lambda) \right), \quad
\theta^*(\lambda) = \arg\min_\theta \mathcal{L}_{\text{train}}(\theta; \lambda)
$$

### Symbols and parameters (word-by-word style)
- $\lambda$: hyperparameter vector (learning rate, depth, dropout, etc.).
- $\Lambda$: hyperparameter search space.
- $\mathcal{M}^{(k)}$: validation metric on fold $k$.
- $\mathcal{V}(\lambda)$: cross-validated objective for tuning.
- $\theta^*(\lambda)$: trained model parameters under $\lambda$.

### Assumptions
- Cross-validation metric approximates leaderboard target.
- Search space includes performant regions.
- Training process is sufficiently reproducible for fair comparisons.

### Why used / effect on model quality
- Separates model fitting from configuration selection in a principled bilevel form.
- Increases probability of near-optimal settings versus manual guessing.
- Usually improves both mean score and variance across folds.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Bergstra and Bengio (2012) random search; practical AutoML formulations.
- Practical takeaway: formalizing tuning as optimization improves experimentation discipline and reproducibility.

### Mathematical advantages
- Explicit objective for search algorithms (random, Bayesian, evolutionary).
- Naturally supports multi-fold uncertainty estimates.
- Extensible to constrained and multi-objective tuning.

### Mathematical disadvantages / failure modes
- Computationally expensive nested optimization.
- Susceptible to validation overfitting with too many trials.
- Objective noise from stochastic training can mislead search.


## Formula 06.2 — TPE acquisition criterion used by Optuna

### Formal definition
$$x^*=\arg\max_x\frac{l(x)}{g(x)},\quad l(x)=p(x\mid y<y^*),\quad g(x)=p(x\mid y\ge y^*),\quad y^*=\text{quantile}_\gamma(\{y_i\}).$$

### Symbols and parameters (word-by-word style)
- $x$: candidate hyperparameter value (or vector component).
- $y$: observed objective value from a trial.
- $y^*$: threshold separating good and bad trials.
- $l(x)$: density of promising region.
- $g(x)$: density of non-promising region.
- $\gamma$: quantile fraction controlling exploration/exploitation.

### Assumptions
- Density estimators for $l(x)$ and $g(x)$ are well-calibrated.
- Historical trial set is informative.
- Objective values are comparable across trials.

### Why used / effect on model quality
- Prioritizes candidates likely to improve best score with fewer evaluations.
- More sample-efficient than uniform random in many tabular tuning tasks.
- Often reaches competitive regions quickly under fixed compute budgets.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Bergstra et al. (2011) Tree-structured Parzen Estimator.
- Practical takeaway: TPE is strong for mixed discrete/continuous search spaces common in Kaggle workflows.

### Mathematical advantages
- Works with conditional and tree-structured spaces.
- Nonparametric density ratio avoids explicit surrogate of $p(y\mid x)$.
- Naturally balances exploitation and exploration via $\gamma$.

### Mathematical disadvantages / failure modes
- Density estimates can be unstable with few trials.
- High-dimensional spaces degrade KDE quality.
- Can over-exploit early lucky regions when objective noise is high.


## Formula 07.1 — Additive gradient boosting update

### Formal definition
$$\hat{y}^{(t)}(x)=\hat{y}^{(t-1)}(x)+\eta f_t(x),\quad f_t\in\mathcal{F}_{trees}. $$

### Symbols and parameters (word-by-word style)
- $\hat{y}^{(t)}$: ensemble prediction after boosting round $t$.
- $f_t$: tree added at round $t$.
- $\eta$: shrinkage (learning-rate) parameter.
- $\mathcal{F}_{trees}$: hypothesis class of regression trees.

### Assumptions
- Weak learners can approximate negative-gradient directions.
- Additive residual fitting remains valid under chosen loss.
- Boosting rounds are regularized (depth, leaves, subsampling).

### Why used / effect on model quality
- Sequentially corrects residual errors left by prior trees.
- Delivers strong performance on heterogeneous tabular features.
- Shrinkage-depth trade-off controls bias and variance in leaderboard tasks.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Friedman (2001) gradient boosting machines.
- Practical takeaway: small $\eta$ with enough rounds usually improves generalization but needs careful early stopping.

### Mathematical advantages
- Flexible function approximation with nonlinearity and interactions.
- Handles mixed feature types with minimal preprocessing.
- Strong empirical risk minimization behavior on tabular data.

### Mathematical disadvantages / failure modes
- Overfitting risk with deep trees or too many rounds.
- Sensitive to leakage and target leakage in engineered features.
- Sequential dependency reduces parallelism relative to bagging.


## Formula 07.2 — Second-order split objective (XGBoost-style)

### Formal definition
$$
\mathcal{L}^{(t)} \approx \sum_i \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2 \right] + \Omega(f_t), \quad
\Omega(f_t) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

$$
w_j^* = - \frac{G_j}{H_j + \lambda}, \quad
\mathrm{Gain} = \frac{1}{2} \left( \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G^2}{H + \lambda} \right) - \gamma
$$
### Symbols and parameters (word-by-word style)
- $g_i,h_i$: first and second derivatives of loss at sample $i$.
- $T$: number of leaves in tree $f_t$.
- $w_j$: score assigned to leaf $j$.
- $G_j=\sum_{i\in I_j}g_i$, $H_j=\sum_{i\in I_j}h_i$: aggregated gradients/Hessians per leaf.
- $\gamma,\lambda$: complexity and L2 leaf penalties.

### Assumptions
- Second-order Taylor expansion is locally accurate.
- Hessians are non-negative or effectively stabilized.
- Greedy split search approximates global tree optimum.

### Why used / effect on model quality
- Uses curvature information for better step sizing than first-order boosting.
- Gain formula provides a rigorous split criterion with complexity penalty.
- Usually improves tabular accuracy and robustness under noisy gradients.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Chen and Guestrin (2016) XGBoost.
- Practical takeaway: second-order regularized gain is a core reason XGBoost remains a strong competition baseline.

### Mathematical advantages
- Closed-form optimal leaf weights.
- Principled complexity control via $\gamma,\lambda$.
- Efficient histogram-based implementations preserve this objective structure.

### Mathematical disadvantages / failure modes
- Hessian approximations can be noisy for non-smooth objectives.
- Greedy splits may miss globally better structures.
- Large search spaces still require careful regularization and validation.


## Formula 08.1 — Feature-map composition for tabular modeling

### Formal definition
$$\hat{y}=f_\theta(\phi(x)).$$

### Symbols and parameters (word-by-word style)
- $x$: raw feature vector.
- $\phi(\cdot)$: engineered feature transformation pipeline.
- $f_\theta$: downstream model (MLP/GBDT/etc.).
- $\hat{y}$: prediction after transformation plus modeling.

### Assumptions
- The same transformation $\phi$ is applied consistently in train/validation/test.
- Engineered features are leakage-safe.
- Feature generation is computationally feasible at inference.

### Why used / effect on model quality
- Encodes domain priors directly into representational space.
- Can convert difficult nonlinear patterns into easier decision surfaces.
- Strong feature maps often improve metric more than switching model family.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: representation/feature map perspective from statistical learning and kernel methods.
- Practical takeaway: disciplined feature engineering remains a top leverage point in tabular competitions.

### Mathematical advantages
- Separates representation design from estimator optimization.
- Enables targeted ablations on transformation blocks.
- Can reduce approximation error for simple models.

### Mathematical disadvantages / failure modes
- Feature explosion increases variance and sample complexity.
- Handcrafted mappings can be brittle under covariate shift.
- Leakage-prone encodings can produce misleading validation gains.


## Formula 08.2 — Bias-variance-noise decomposition

### Formal definition
$$
\mathbb{E}\left[(y - \hat{y})^2 \right] = \mathrm{Bias}[\hat{y}]^2 + \mathrm{Var}[\hat{y}] + \sigma^2
$$

### Symbols and parameters (word-by-word style)
- $y$: target variable.
- $\hat{y}$: model prediction.
- $\mathrm{Bias}[\hat{y}]$: systematic prediction error.
- $\mathrm{Var}[\hat{y}]$: variability across training samples.
- $\sigma^2$: irreducible noise in data generation.

### Assumptions
- Squared-error loss setting.
- Expectation taken over repeated training-set draws.
- Noise term is independent of model randomness.

### Why used / effect on model quality
- Frames why stronger models can still generalize worse.
- Guides trade-offs between capacity, regularization, and ensembling.
- Helps interpret fold instability and overfitting symptoms.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Geman, Bienenstock, Doursat (1992) bias-variance analysis.
- Practical takeaway: competition gains often come from variance reduction after bias is reasonably controlled.

### Mathematical advantages
- Clear decomposition for diagnosis of generalization error.
- Supports principled decisions on complexity and data augmentation.
- Connects naturally to ensemble variance reduction formulas.

### Mathematical disadvantages / failure modes
- Exact decomposition is loss-specific (cleanest under MSE).
- Hard to estimate components precisely in single finite datasets.
- Misspecification and shift can blur the diagnostic signal.


## Formula 08.3 — Smoothed target encoding

### Formal definition
$$\tilde{\mu}_c=\frac{n_c\mu_c+\alpha\mu_{global}}{n_c+\alpha}. $$

### Symbols and parameters (word-by-word style)
- $c$: categorical level.
- $n_c$: count of rows with category $c$.
- $\mu_c$: category-specific target mean.
- $\mu_{global}$: global target mean.
- $\alpha$: smoothing strength (pseudo-count).
- $\tilde{\mu}_c$: smoothed encoded value.

### Assumptions
- Category-level means carry predictive signal.
- Encoding is computed in leakage-safe CV folds.
- Smoothing constant reflects data sparsity.

### Why used / effect on model quality
- Retains target signal while reducing variance of rare categories.
- Strongly beneficial for high-cardinality categorical features.
- Proper smoothing can improve both convergence and leaderboard stability.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: Micci-Barreca (2001) high-cardinality preprocessing; broad Kaggle practice.
- Practical takeaway: out-of-fold target encoding with smoothing is often superior to one-hot for rare categories.

### Mathematical advantages
- Bayesian shrinkage interpretation toward prior mean.
- Handles rare categories without extreme estimates.
- Compact numeric representation for tree and neural models.

### Mathematical disadvantages / failure modes
- Leakage if computed on full data before split.
- Over-smoothing can erase genuine category signal.
- Under-smoothing causes noisy, high-variance encodings.


## Formula 09.1 — Competition objective and CV proxy metric

### Formal definition
$$\theta^*=\arg\min_{\theta\in\Theta}\mathbb{E}[\mathcal{M}_{private}(\theta)],\quad \widehat{\mathcal{M}}_{CV}(\theta)=\frac{1}{K}\sum_{k=1}^{K}\mathcal{M}^{(k)}(\theta),\quad \Delta_{shake}=\mathcal{M}_{public}-\mathcal{M}_{private}.$$

### Symbols and parameters (word-by-word style)
- $\Theta$: admissible model/hyperparameter space.
- $\mathcal{M}_{private}$: hidden leaderboard metric on private holdout.
- $\widehat{\mathcal{M}}_{CV}$: cross-validation estimate of expected competition performance.
- $\mathcal{M}_{public}$: public leaderboard metric.
- $\Delta_{shake}$: public-private discrepancy (leaderboard shake).

### Assumptions
- CV folds mimic competition distribution and scoring protocol.
- Public and private splits are sampled from related processes.
- Metric implementation matches competition definition exactly.

### Why used / effect on model quality
- Reorients optimization to true objective: private leaderboard expectation.
- Encourages robust validation strategy over public-board overfitting.
- Reduces late-stage rank collapse from unstable model selection.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: cross-validation risk estimation literature and Kaggle best-practice playbooks.
- Practical takeaway: teams with disciplined CV pipelines are more resilient to leaderboard shake.

### Mathematical advantages
- Explicitly models hidden-test uncertainty.
- Provides quantitative guardrail against public-score chasing.
- Supports confidence intervals across folds/repeats.

### Mathematical disadvantages / failure modes
- CV can still be biased under temporal or group leakage.
- Small validation sets inflate estimator variance.
- Public leaderboard feedback loops induce adaptive overfitting.


## Formula 09.2 — Weighted ensembling and error-covariance optimum

### Formal definition
$$\hat{y}_{ens}=w\hat{y}_1+(1-w)\hat{y}_2,\quad w\in[0,1],$$
$$w^*=\frac{\sigma_2^2-ho\sigma_1\sigma_2}{\sigma_1^2+\sigma_2^2-2ho\sigma_1\sigma_2},\quad \mathrm{Var}(e_{ens})=w^2\sigma_1^2+(1-w)^2\sigma_2^2+2w(1-w)ho\sigma_1\sigma_2.$$

### Symbols and parameters (word-by-word style)
- $\hat{y}_1,\hat{y}_2$: predictions from base models.
- $w$: blend weight for model 1.
- $e_{ens}$: ensemble prediction error.
- $\sigma_1^2,\sigma_2^2$: error variances of base models.
- $ho$: correlation between model errors.
- $w^*$: variance-minimizing blend weight.

### Assumptions
- Base predictors are approximately unbiased for target.
- Error variances/correlation estimated reliably from validation.
- Linear blend is acceptable for metric behavior.

### Why used / effect on model quality
- Exploits diversity between models to reduce error variance.
- Usually yields more stable private scores than any single model.
- Weight tuning can recover performance even when one model is weaker alone.

### Research context (canonical paper/source + practical takeaway)
- Canonical source: ensemble theory (Hansen and Salamon, 1990) and stacked generalization (Wolpert, 1992).
- Practical takeaway: diversity (low error correlation) is often more valuable than marginal single-model score differences.

### Mathematical advantages
- Closed-form optimum for two-model MSE blending.
- Transparent bias-variance-covariance interpretation.
- Extends to constrained quadratic programming for many models.

### Mathematical disadvantages / failure modes
- Correlation estimates are noisy under limited validation data.
- Linear blending cannot capture nonlinear complementarity.
- Over-tuned weights on public leaderboard can overfit and shake.
